# PlantEye F500 Standalone Colab Tutorial

This notebook is designed for people who want to inspect PlantEye F500 data in Google Colab without editing much code.

It uses only two CSV files from the PlantEye/Derived export:

1. <EXPERIMENT_ID>_measured_traits.csv
2. <EXPERIMENT_ID>_averages_of_all_indices.csv

Most users only need to upload those two CSV files, set the folder path in DATA_ROOT, set EXPERIMENT_ID, and then run the notebook from top to bottom.

## PlantEye F500 Technology Overview

The PlantEye F500 is an advanced 3D multispectral plant scanner developed by Phenospex. It captures plant traits in a non-invasive way and is often used in high-throughput greenhouse phenotyping.

The exported derived CSV files already contain useful measurements such as plant height, digital biomass, leaf angle, and vegetation index averages. This notebook focuses on exploring those ready-to-use values rather than processing raw point clouds.


## Setup

This notebook needs exactly two CSV files. The easiest and most reliable Colab workflow is to upload them to Google Drive first.

### Recommended: upload through Google Drive

1. Open Google Drive in your browser.
2. Create a folder named M5 Colab.
3. Inside that folder, create a folder named derived.
4. Upload these two CSV files into that derived folder:
   - 46_measured_traits.csv
   - 46_averages_of_all_indices.csv
5. In Colab, run the first setup code cell. It mounts Google Drive automatically.
6. In the next setup code cell, set DATA_ROOT to the folder that directly contains the two CSV files.

For the folder above, DATA_ROOT should be:

/content/drive/MyDrive/M5 Colab/derived

The complete file paths should then look like this:

/content/drive/MyDrive/M5 Colab/derived/46_measured_traits.csv
/content/drive/MyDrive/M5 Colab/derived/46_averages_of_all_indices.csv

If your files start with another number, change EXPERIMENT_ID. For example, if the files are 41_measured_traits.csv and 41_averages_of_all_indices.csv, set EXPERIMENT_ID = 41.

### Temporary alternative: upload directly into Colab

Use this only for quick testing. Files uploaded directly into Colab disappear when the Colab session resets.

1. Click the folder icon on the left side of Colab.
2. Click the upload button.
3. Upload both CSV files.
4. Set DATA_ROOT = Path("/content").
5. Keep EXPERIMENT_ID matching the number at the start of the CSV filenames.

Optional settings:

- threshold_date: set this if you want to remove trial scans before the real experiment started.
- INCLUDE_EMPTY_POTS: keep this False unless you want to include empty pot rows.


In [ ]:
import pandas as pd
from pathlib import Path
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import numpy as np
import seaborn as sns
from IPython.display import Markdown, display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

if IN_COLAB and drive is not None:
    drive.mount("/content/drive")

pd.options.display.max_columns = 80
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


In [ ]:
# USER SETTINGS: change only these two lines for a normal Colab run.
#
# DATA_ROOT must be the folder that directly contains the two CSV files.
# Do not include the filename here.
#
# Recommended Google Drive example:
# DATA_ROOT = Path("/content/drive/MyDrive/M5 Colab/derived")
#
# If you uploaded the CSV files directly through Colab's left file panel:
# DATA_ROOT = Path("/content")
#
# EXPERIMENT_ID is the number at the start of the filenames.
# Example: 46_measured_traits.csv means EXPERIMENT_ID = 46.

DATA_ROOT = Path("/content/drive/MyDrive/M5 Colab/derived") if IN_COLAB else Path(r"D:\Maarten\M5\Data")
EXPERIMENT_ID = 46

# The notebook builds the exact CSV paths from DATA_ROOT and EXPERIMENT_ID.
MEASURED_TRAITS_CSV = DATA_ROOT / f"{EXPERIMENT_ID}_measured_traits.csv"
AVERAGE_INDICES_CSV = DATA_ROOT / f"{EXPERIMENT_ID}_averages_of_all_indices.csv"

# Optional settings.
threshold_date = None  # Example: "2026-04-13" to skip earlier trial scans.
INCLUDE_EMPTY_POTS = False
MAX_SAMPLE_NAMES_TO_SHOW = 40
TIMESTAMP_FORMAT = "%Y%m%dT%H%M%S"

print("The notebook will read these files:")
print(f"- {MEASURED_TRAITS_CSV}")
print(f"- {AVERAGE_INDICES_CSV}")


In [ ]:
def read_planteye_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Check that DATA_ROOT is the folder containing the two CSV files."
        )
    return pd.read_csv(path, sep=";", decimal=",")


def position_number(position):
    match = re.search(r"(\d+)$", str(position))
    return match.group(1) if match else str(position)


def parse_sample_metadata(sample):
    parts = str(sample).split(".")
    result = {
        "ExperimentName": pd.NA,
        "ExperimentDate": pd.NA,
        "Position": pd.NA,
        "PositionNumber": pd.NA,
        "Genotype": pd.NA,
        "Treatment": pd.NA,
        "Replicate": pd.NA,
        "SampleName": str(sample),
        "GroupName": pd.NA,
    }
    if len(parts) >= 2:
        result["ExperimentName"] = f"{parts[0]}.{parts[1]}"
        result["ExperimentDate"] = parts[1]
    if len(parts) >= 6:
        result["Position"] = parts[2]
        result["PositionNumber"] = position_number(parts[2])
        result["Genotype"] = parts[3]
        result["Treatment"] = parts[4]
        try:
            result["Replicate"] = int(parts[5])
        except ValueError:
            result["Replicate"] = parts[5]
        result["SampleName"] = f"{result['PositionNumber']}.{parts[3]}.{parts[4]}.{parts[5]}"
        result["GroupName"] = f"{parts[3]}.{parts[4]}"
    return result


def add_sample_metadata(data, sample_column="sample"):
    if sample_column not in data.columns:
        return data
    result = data.copy()
    parsed = pd.DataFrame([parse_sample_metadata(value) for value in result[sample_column]], index=result.index)
    for column in parsed.columns:
        result[column] = parsed[column]
    result["Pot"] = result[sample_column]
    result["IsEmptyPot"] = result["Genotype"].eq("empty") | result["Treatment"].eq("empty")
    return result


def prepare_time_columns(data):
    result = data.copy()
    if "timestamp" not in result.columns:
        raise KeyError("The CSV needs a timestamp column.")
    result["timestamp_text"] = result["timestamp"].astype(str)
    result["timestamp"] = pd.to_datetime(result["timestamp"], format=TIMESTAMP_FORMAT, errors="coerce")
    result = result.dropna(subset=["timestamp"]).copy()
    result["day"] = result["timestamp"].dt.date
    result["timestamp_hourly"] = result["timestamp"].dt.floor("h")
    return result


## Read The Two CSV Files

This section reads only the two useful derived CSV files:

- measured traits: plant height, biomass, leaf traits, 3D area, projected area, and related structural measurements
- average indices: average vegetation index values such as NDVI, greenness, hue, NPCI, and PSRI when present

If this cell fails with FileNotFoundError, the two most common causes are:

- DATA_ROOT points to the wrong folder. It must point to the folder that directly contains the CSV files.
- EXPERIMENT_ID does not match the number at the start of the filenames.


In [ ]:
measured_traits = read_planteye_csv(MEASURED_TRAITS_CSV)
average_indices = read_planteye_csv(AVERAGE_INDICES_CSV)

# In the PlantEye CSV, Treatment is a scan/system column. The biological treatment is parsed from sample.
if "Treatment" in measured_traits.columns and "ScanTreatmentId" not in measured_traits.columns:
    measured_traits = measured_traits.rename(columns={"Treatment": "ScanTreatmentId"})
if "Treatment" in average_indices.columns and "ScanTreatmentId" not in average_indices.columns:
    average_indices = average_indices.rename(columns={"Treatment": "ScanTreatmentId"})

phenotypic_data = add_sample_metadata(measured_traits)
average_indices = add_sample_metadata(average_indices)

if not INCLUDE_EMPTY_POTS:
    phenotypic_data = phenotypic_data[~phenotypic_data["IsEmptyPot"]].reset_index(drop=True)
    average_indices = average_indices[~average_indices["IsEmptyPot"]].reset_index(drop=True)

print("Files loaded:")
print(f"- {MEASURED_TRAITS_CSV}")
print(f"- {AVERAGE_INDICES_CSV}")
print()
print(f"Measured trait rows: {len(phenotypic_data):,}")
print(f"Average index rows: {len(average_indices):,}")
print(f"Samples: {phenotypic_data['SampleName'].nunique():,}")

display(phenotypic_data.head())

## Format Data And Shorten Sample Names

The next cell prepares timestamps, merges the average vegetation index columns into the measured traits table, and prints the experiment names found in the sample IDs.

This is also where the long PlantEye sample names are shortened. The code reads the original sample text, splits it at the dots, and creates a shorter SampleName from:

position number + genotype + treatment + replicate

Examples:

- NPEC33.20260330.LU32.BM.D_JA.15 becomes 32.BM.D_JA.15
- NPEC33.20260330.LU31.BU.W_JA.14 becomes 31.BU.W_JA.14

By default, the notebook uses all samples. If you want to compare only a few samples, edit the optional example cell later in the notebook.


In [ ]:
def concatenate_columns(row, x, y):
    return f"{row[x]}, {row[y]}"

phenotypic_data = prepare_time_columns(phenotypic_data)
average_indices = prepare_time_columns(average_indices)

if threshold_date is not None:
    threshold = pd.to_datetime(threshold_date)
    phenotypic_data = phenotypic_data[phenotypic_data["timestamp"] >= threshold].copy()
    average_indices = average_indices[average_indices["timestamp"] >= threshold].copy()

metadata_columns = {
    "timestamp", "timestamp_text", "day", "timestamp_hourly", "sample", "Pot",
    "Experiment", "ExperimentName", "ExperimentDate", "Position", "PositionNumber",
    "Genotype", "Treatment", "Replicate", "SampleName", "GroupName", "IsEmptyPot",
    "ScanTreatmentId",
}
average_index_columns = [
    column for column in average_indices.columns
    if column not in metadata_columns and column not in phenotypic_data.columns
]

phenotypic_data = phenotypic_data.merge(
    average_indices[["timestamp", "sample"] + average_index_columns],
    on=["timestamp", "sample"],
    how="left",
)

phenotypic_data["Treatment, genotype"] = phenotypic_data.apply(
    concatenate_columns,
    axis=1,
    x="Treatment",
    y="Genotype",
)

metadata = phenotypic_data[
    ["sample", "SampleName", "ExperimentName", "GroupName", "Position", "Genotype", "Treatment", "Replicate"]
].drop_duplicates().reset_index(drop=True)

experiment_names = sorted(metadata["ExperimentName"].dropna().astype(str).unique())
sample_names = sorted(metadata["SampleName"].dropna().astype(str).unique())

conversion_examples = metadata[["sample", "SampleName"]].drop_duplicates().head(8)
print("Sample name conversion examples:")
display(conversion_examples)
print("Experiment names found in the sample IDs:")
for experiment_name in experiment_names:
    print(f"- {experiment_name}")

print("\nShort sample names found:")
for sample_name in sample_names[:MAX_SAMPLE_NAMES_TO_SHOW]:
    print(f"- {sample_name}")
if len(sample_names) > MAX_SAMPLE_NAMES_TO_SHOW:
    print(f"... and {len(sample_names) - MAX_SAMPLE_NAMES_TO_SHOW} more samples")

# Use every sample by default. To compare only a few samples, edit the optional
# example cell later in the notebook after this cell has run.
selected_sample_names = sample_names
selected_data = phenotypic_data[phenotypic_data["SampleName"].isin(selected_sample_names)].copy()
selected_metadata = metadata[metadata["SampleName"].isin(selected_sample_names)].copy()

print(f"\nSelected samples: {selected_data['SampleName'].nunique():,}")
display(selected_metadata.head(20))

important_columns = [
    "SampleName", "sample", "timestamp", "day", "Position", "Genotype", "Treatment", "Replicate",
    "height", "height_max", "area_3d", "leaf_area_index", "digital_biomass",
    "leaf_angle", "leaf_inclination", "proj_area", "light_pen_depth",
] + average_index_columns
important_columns = [column for column in important_columns if column in selected_data.columns]
selected_data[important_columns].head()


## Summary

This summary checks that the two CSV files were read correctly and gives a quick overview of the selected samples.


In [ ]:
def format_value(value, digits=2):
    if pd.isna(value):
        return "n/a"
    if isinstance(value, (int, float, np.integer, np.floating)):
        return f"{value:.{digits}f}"
    return str(value)


height_column = "height_max" if "height_max" in selected_data.columns else "height"
height_values = selected_data[height_column].dropna() if height_column in selected_data.columns else pd.Series(dtype="float64")

num_data_points = len(selected_data["timestamp"])
genotypes = sorted(selected_data["Genotype"].dropna().astype(str).unique())
num_genotypes = len(genotypes)
max_height = height_values.max() if not height_values.empty else pd.NA
min_max_height = height_values.min() if not height_values.empty else pd.NA
avg_max_height = height_values.mean() if not height_values.empty else pd.NA
treatments = sorted(selected_data["Treatment"].dropna().astype(str).unique())
start_date = selected_data["timestamp"].min()
end_date = selected_data["timestamp"].max()
samples = selected_data["SampleName"].dropna().unique()

summary_text = f"""
### Phenotypic Dataset Summary

- **Total number of measurements:** {num_data_points}
- **Number of selected samples:** {len(samples)}
- **Number of unique genotypes:** {num_genotypes}
- **List of genotypes:** {', '.join(genotypes)}
- **Treatments applied:** {', '.join(treatments)}
- **Height column summarized:** {height_column}
- **Maximum recorded plant height:** {format_value(max_height)} mm
- **Minimum of maximum plant heights:** {format_value(min_max_height)} mm
- **Average of maximum plant heights:** {format_value(avg_max_height)} mm
- **Measurement period:** From **{start_date.strftime('%Y-%m-%d %H:%M')}** to **{end_date.strftime('%Y-%m-%d %H:%M')}**
"""

display(Markdown(summary_text))


## 🌿 **Explanation of Measured Traits from PlantEye F500**

The **PlantEye F500** uses 3D laser scanning and multispectral imaging to non-destructively assess various morphological and physiological traits of plants. The following traits are derived from the **3D point clouds** generated during each scan:

---

### 📏 **1. Height (`height`)**

* **What it is:** The vertical distance from the ground to the highest point of the plant canopy.
* **How it's measured:** Calculated directly from the 3D point cloud by identifying the highest z-coordinate in the plant's scanned area.
* **Why it's useful:** A primary indicator of plant growth over time and response to environmental conditions or treatments.

---

### 🌱 **2. Digital Biomass (`digital_biomass`)**

* **What it is:** An estimate of plant biomass based on the volume and density of the 3D point cloud.
* **How it's measured:** Derived using the number and distribution of 3D points that represent the plant. It often correlates with actual biomass measured destructively.
* **Why it's useful:** Provides a non-destructive approximation of total plant growth, which is crucial for longitudinal studies and high-throughput phenotyping.

---

### 📐 **3. Leaf Inclination (`leaf_inclination`)**

* **What it is:** The average **tilt angle** of the leaves relative to the vertical axis.
* **How it's measured:** Calculated using the orientation of 3D surface normals in the point cloud, which indicate how steeply the leaves are angled upward or outward.
* **Why it's useful:** Reflects the plant's light interception strategy. More upright leaves can indicate different photosynthetic or stress-adaptive behavior.

---

### 🔄 **4. Leaf Angle (`leaf_angle`)**

* **What it is:** Often refers to the angle between a leaf surface and the horizontal plane.
* **How it's measured:** Similar to leaf inclination, but this metric may average angles more broadly across the canopy or use a different reference axis.
* **Why it's useful:** A sensitive indicator of plant response to environmental stresses (e.g., drought or light competition), as leaf posture can change in reaction to water availability or light direction.

---

### 🔬 **Why These Traits Matter**

These measurements together provide a **multi-dimensional view of plant growth**:

* **Height and digital biomass** tell you about **size and structural development**.
* **Leaf inclination and angle** tell you about **architecture and physiological responses**.

All are captured automatically and non-destructively by the PlantEye F500, enabling repeated measurements over time with high throughput.



## Plotting Functions

These functions make cleaner multi-panel figures for the selected data. They keep the old function names from the original tutorial, but the plots are more compact and easier to compare.


In [ ]:
TRAIT_LABELS = {
    "digital_biomass": "Digital biomass",
    "height": "Height",
    "height_max": "Maximum height",
    "leaf_inclination": "Leaf inclination",
    "leaf_angle": "Leaf angle",
    "area_3d": "3D area",
    "leaf_area_index": "Leaf area index",
    "proj_area": "Projected area",
    "light_pen_depth": "Light penetration depth",
}


def pretty_label(column):
    return TRAIT_LABELS.get(column, str(column).replace("_", " ").title())


def _available_traits(data, traits):
    available = [trait for trait in traits if trait in data.columns]
    missing = [trait for trait in traits if trait not in data.columns]
    if missing:
        print(f"Skipping missing traits: {', '.join(missing)}")
    if not available:
        raise ValueError("None of the requested traits are present in the data.")
    return available


def _palette_map(data, hue):
    if hue not in data.columns:
        return {}, []
    values = sorted(data[hue].dropna().astype(str).unique())
    palette = sns.color_palette("tab20", n_colors=max(len(values), 1))
    return dict(zip(values, palette)), values


def _format_time_axis(ax):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=7))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    ax.tick_params(axis="x", rotation=0)


def _line_panel(data, traits, hue="SampleName", title=None, legend=True, max_legend_items=16):
    if data.empty:
        print("No rows available for plotting.")
        return None

    traits = _available_traits(data, traits)
    plot_data = data.sort_values("timestamp_hourly").copy()
    palette, hue_values = _palette_map(plot_data, hue)
    if hue in plot_data.columns:
        plot_data[hue] = plot_data[hue].astype(str)

    ncols = min(2, len(traits))
    nrows = int(np.ceil(len(traits) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.2 * ncols, 3.8 * nrows), sharex=True, squeeze=False)
    axes_flat = axes.ravel()

    for ax, trait in zip(axes_flat, traits):
        sns.lineplot(
            data=plot_data,
            x="timestamp_hourly",
            y=trait,
            hue=hue if hue in plot_data.columns else None,
            palette=palette if palette else None,
            marker="o",
            markersize=4,
            linewidth=2,
            alpha=0.9,
            errorbar=None,
            legend=False,
            ax=ax,
        )
        ax.set_title(pretty_label(trait))
        ax.set_xlabel("")
        ax.set_ylabel(pretty_label(trait))
        ax.grid(True, alpha=0.28)
        _format_time_axis(ax)

    for ax in axes_flat[len(traits):]:
        ax.set_visible(False)

    if title:
        fig.suptitle(title, fontsize=15, fontweight="bold", y=1.02)

    if legend and hue_values:
        shown_values = hue_values[:max_legend_items]
        handles = [
            Line2D([0], [0], color=palette[value], marker="o", linewidth=2, label=value)
            for value in shown_values
        ]
        legend_title = hue if len(hue_values) <= max_legend_items else f"{hue} (first {max_legend_items})"
        fig.legend(handles=handles, title=legend_title, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)

    plt.tight_layout()
    plt.show()
    return fig


def plotTraitOverview(data=selected_data, hue="SampleName", legend=True):
    traits = ["digital_biomass", "height", "height_max", "leaf_inclination", "leaf_angle", "area_3d"]
    return _line_panel(data, traits, hue=hue, title="Plant trait overview", legend=legend)


def plotTreatment(Experiment, Treatment, hue="SampleName", legend=True):
    Treatment = list(Treatment)
    traits = ["digital_biomass", "height", "leaf_inclination", "leaf_angle"]
    filtered = Experiment[Experiment["Treatment"].isin(Treatment)].copy()
    return _line_panel(filtered, traits, hue=hue, title="Traits grouped by treatment", legend=legend)


def plotGenotypes(Experiment, Genotypes, hue="Genotype", legend=True):
    Genotypes = list(Genotypes)
    traits = ["digital_biomass", "height", "leaf_inclination", "leaf_angle"]
    filtered = Experiment[Experiment["Genotype"].isin(Genotypes)].copy()
    return _line_panel(filtered, traits, hue=hue, title="Traits grouped by genotype", legend=legend)


def plotGenotypePanel(Experiment, Genotypes, hue="Treatment", legend=True, traits=["digital_biomass", "height", "leaf_inclination", "leaf_angle"]):
    Genotypes = list(Genotypes)
    traits = _available_traits(Experiment, traits)
    filtered = Experiment[Experiment["Genotype"].isin(Genotypes)].sort_values("timestamp_hourly").copy()
    if filtered.empty:
        print("No rows available for the selected genotypes.")
        return None

    palette, hue_values = _palette_map(filtered, hue)
    if hue in filtered.columns:
        filtered[hue] = filtered[hue].astype(str)

    fig, axes = plt.subplots(
        nrows=len(Genotypes),
        ncols=len(traits),
        figsize=(5.4 * len(traits), 3.4 * len(Genotypes)),
        sharex=True,
        sharey="col",
        squeeze=False,
    )

    for row_idx, genotype in enumerate(Genotypes):
        genotype_data = filtered[filtered["Genotype"] == genotype]
        for col_idx, trait in enumerate(traits):
            ax = axes[row_idx, col_idx]
            sns.lineplot(
                data=genotype_data,
                x="timestamp_hourly",
                y=trait,
                hue=hue if hue in genotype_data.columns else None,
                palette=palette if palette else None,
                marker="o",
                markersize=4,
                linewidth=2,
                errorbar=None,
                legend=False,
                ax=ax,
            )
            if row_idx == 0:
                ax.set_title(pretty_label(trait))
            ax.set_ylabel(str(genotype) if col_idx == 0 else "")
            ax.set_xlabel("")
            ax.grid(True, alpha=0.28)
            _format_time_axis(ax)

    fig.suptitle("Genotype comparison", fontsize=15, fontweight="bold", y=1.02)

    if legend and hue_values:
        handles = [Line2D([0], [0], color=palette[value], marker="o", linewidth=2, label=value) for value in hue_values]
        fig.legend(handles=handles, title=hue, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)

    plt.tight_layout()
    plt.show()
    return fig


## Plot Traits Grouped By Treatment

This shows how selected plants change over time within each treatment.


In [ ]:
plotTreatment(selected_data, selected_data["Treatment"].unique(), hue="SampleName", legend=False)
plotTraitOverview(selected_data, hue="Treatment", legend=True)


## Plot Traits Grouped By Genotype

These plots compare selected plants by genotype and treatment.


In [ ]:
plotGenotypePanel(selected_data, selected_data["Genotype"].unique(), hue="Treatment", legend=True, traits=["leaf_inclination", "leaf_angle"])


In [ ]:
plotGenotypes(selected_data, selected_data["Genotype"].unique(), hue="Treatment")


## 🌾 Vegetation Indices Provided by PlantEye F500

Vegetation indices are mathematical combinations of spectral reflectance values (typically in the Red, Green, Blue, and Near-Infrared wavelengths) that provide insights into plant physiology, health, and stress status. The following indices are calculated directly from the multispectral data captured by the PlantEye F500:

---

### 🌿 1. **Greenness Index**
- **Purpose:** Measures the relative contribution of green light reflectance to overall reflectance, indicating general "greenness" of vegetation.
- **Formula:**  
  \[
  \text{Greenness} = \frac{G}{R + G + B}
  \]
- **Where:**
  - \( G \): Green reflectance
  - \( R \): Red reflectance
  - \( B \): Blue reflectance
- **Interpretation:** Higher values generally correspond to healthy, green plant tissue.

---

### 🌱 2. **Normalized Difference Vegetation Index (NDVI)**
- **Purpose:** A widely used index for estimating vegetation health and biomass.
- **Formula:**  
  \[
  \text{NDVI} = \frac{NIR - R}{NIR + R}
  \]
- **Where:**
  - \( NIR \): Near-Infrared reflectance
  - \( R \): Red reflectance
- **Interpretation:** Values range from -1 to 1.
  - Healthy vegetation typically has NDVI > 0.5
  - Bare soil or stressed vegetation has NDVI near 0 or negative

---

### 🍂 3. **Plant Senescence Reflectance Index (PSRI)**
- **Purpose:** Detects leaf senescence (aging), linked to pigment degradation (e.g., chlorophyll breakdown).
- **Formula:**  
  \[
  \text{PSRI} = \frac{R - G}{NIR}
  \]
- **Where:**
  - \( R \): Red reflectance
  - \( G \): Green reflectance
  - \( NIR \): Near-Infrared reflectance
- **Interpretation:** Higher values indicate increased senescence.

---

### 🎨 4. **Hue**
- **Purpose:** Represents the **dominant color** of the plant surface in HSV (Hue, Saturation, Value) color space.
- **Formula:**  
  Calculated from RGB reflectance values using standard color-space conversion.
- **Interpretation:**  
  Hue values range from 0–360°, where:
  - Green ~ 90–150°
  - Yellow/Orange ~ 30–60°
  - Red ~ 0–30° or 330–360°
- Useful for detecting color changes due to stress or senescence.

---

### 🌕 5. **Normalized Pigment Chlorophyll Index (NPCI)**
- **Purpose:** Estimates pigment composition, especially the chlorophyll to carotenoid ratio.
- **Formula:**  
  \[
  \text{NPCI} = \frac{R - B}{R + B}
  \]
- **Where:**
  - \( R \): Red reflectance
  - \( B \): Blue reflectance
- **Interpretation:** Higher NPCI values often indicate chlorophyll loss and increasing carotenoid presence (a sign of stress or aging).

---

### 🧪 Summary Table

| Index     | Indicates                      | Formula                                      |
|-----------|--------------------------------|----------------------------------------------|
| Greenness | Vegetative vigor               | \( \frac{G}{R + G + B} \)                     |
| NDVI      | Biomass, vegetation health     | \( \frac{NIR - R}{NIR + R} \)                |
| PSRI      | Leaf senescence                | \( \frac{R - G}{NIR} \)                      |
| Hue       | Color/hue shift in foliage     | From RGB (HSV conversion)                    |
| NPCI      | Chlorophyll-to-carotenoid ratio| \( \frac{R - B}{R + B} \)                    |

---

These indices are powerful tools for **non-invasive monitoring** of plant condition and development, and they can be used to detect **early stress symptoms** or evaluate treatment effects.



## Average Vegetation Index Data

This section uses the second CSV file, <EXPERIMENT_ID>_averages_of_all_indices.csv.

The exact columns can vary between exports. The function below automatically finds numeric vegetation index columns from that CSV and plots them over time.


In [ ]:
def get_average_index_columns():
    metadata_columns = {
        "timestamp", "timestamp_text", "day", "timestamp_hourly", "sample", "Pot",
        "Experiment", "ExperimentName", "ExperimentDate", "Position", "PositionNumber",
        "Genotype", "Treatment", "Replicate", "SampleName", "GroupName", "IsEmptyPot",
        "ScanTreatmentId",
    }
    return [
        column for column in average_index_columns
        if column in selected_data.columns
        and column not in metadata_columns
        and pd.api.types.is_numeric_dtype(selected_data[column])
    ]


def plotAverageIndices(data, indices=None, hue="SampleName", legend=True):
    if indices is None:
        indices = get_average_index_columns()
    if not indices:
        print("No numeric average index columns were found in the average indices CSV.")
        return None
    return _line_panel(data, indices, hue=hue, title="Average vegetation indices", legend=legend)


def plotIndexCorrelation(data=selected_data, indices=None):
    if indices is None:
        indices = get_average_index_columns()
    if len(indices) < 2:
        print("At least two numeric index columns are needed for a correlation heatmap.")
        return None

    corr = data[indices].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(min(12, 1.1 * len(indices) + 3), min(10, 1.0 * len(indices) + 2)))
    sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", linewidths=0.5, square=True, ax=ax)
    ax.set_title("Correlation between average vegetation indices")
    plt.tight_layout()
    plt.show()
    return fig


print("Average index columns available:")
for column in get_average_index_columns():
    print(f"- {column}")


## Index Plots

These plots show the average vegetation index values over time for the selected samples.


In [ ]:
plotAverageIndices(selected_data, hue="SampleName", legend=True)
plotIndexCorrelation(selected_data)


## Selected Sample Overview

This table is a simple check of the samples currently selected for comparison. Use the short SampleName values in the prompt above if you want to focus on specific samples.


In [ ]:
def show_sample_overview(data=selected_data):
    columns = [
        "SampleName", "sample", "ExperimentName", "GroupName", "Position",
        "Genotype", "Treatment", "Replicate",
    ]
    columns = [column for column in columns if column in data.columns]
    overview = data[columns].drop_duplicates().sort_values("SampleName").reset_index(drop=True)
    display(overview)
    return overview


sample_overview = show_sample_overview(selected_data)


In [ ]:
# Example: select two samples after the notebook is loaded.
# selected_data = phenotypic_data[phenotypic_data["SampleName"].isin(["32.BM.D_JA.15", "31.BU.W_JA.14"])].copy()
# show_sample_overview(selected_data)


## Plant Vigor

Plant vigor is estimated here as the change in digital biomass over time. The notebook uses the first selected sample by default. To inspect another sample, change sample to one of the short sample names, for example "32.BM.D_JA.15".


In [ ]:
# Sample to calculate vigor for. Change this to another short SampleName if needed.
sample = selected_metadata["SampleName"].iloc[0] if not selected_metadata.empty else selected_data["SampleName"].iloc[0]

vigor = selected_data[selected_data["SampleName"] == sample].sort_values(by=["SampleName", "timestamp"])
if vigor.empty:
    raise ValueError(f"No rows found for sample {sample!r}")

resample_interval = "12h"


def interpolate_biomass(df):
    df = df.set_index("timestamp")
    resampled_df = df[["digital_biomass"]].resample(resample_interval).mean()
    resampled_df["digital_biomass"] = resampled_df["digital_biomass"].interpolate(method="linear")
    resampled_df["digital_biomass"] = resampled_df["digital_biomass"].rolling(window=5, min_periods=1).mean()
    resampled_df["SampleName"] = sample
    return resampled_df.reset_index()


df_interpolated = interpolate_biomass(vigor)
df_interpolated["time_change"] = df_interpolated["timestamp"].diff().dt.total_seconds() / 3600.0
df_interpolated["biomass_change"] = df_interpolated["digital_biomass"].diff()
df_interpolated["biomass_change"] = df_interpolated["biomass_change"].interpolate(method="linear")
df_interpolated["biomass_change"] = df_interpolated["biomass_change"].rolling(window=5, min_periods=1).mean()
df_interpolated["vigor"] = df_interpolated["biomass_change"] / df_interpolated["time_change"]
df_interpolated = df_interpolated.dropna(subset=["time_change", "biomass_change"])

plt.plot(df_interpolated["timestamp"], df_interpolated["vigor"], label=sample)
plt.xlabel("Date")
plt.ylabel("Vigor (digital biomass change per hour)")
plt.title(f"Vigor over time for {sample}")
plt.legend(title="Sample")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

plt.plot(df_interpolated["timestamp"], df_interpolated["digital_biomass"], label=sample)
plt.xlabel("Date")
plt.ylabel("Digital biomass")
plt.title(f"Digital biomass over time for {sample}")
plt.legend(title="Sample")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.patches as mpatches

height_for_check = "height" if "height" in selected_data.columns else "height_max"

summary = selected_data.groupby("SampleName").agg(
    Max=(height_for_check, "max"),
    Min=(height_for_check, "min"),
    Position=("Position", "first"),
    PositionNumber=("PositionNumber", "first"),
).reset_index()

summary["LargeHeight"] = summary["Max"] > 200
result = summary[["SampleName", "Position", "PositionNumber", "LargeHeight"]].copy()
display(result)

result["Table"] = result["Position"].astype(str).str[0]
result["Column"] = result["Position"].astype(str).str[1]
result["Row"] = pd.to_numeric(result["PositionNumber"], errors="coerce")
result = result.dropna(subset=["Row"]).copy()
result["Row"] = result["Row"].astype(int)
result = result.sort_values(by=["Table", "Column", "Row"])

for table in result["Table"].unique():
    table_data = result[result["Table"] == table]
    cols = sorted(table_data["Column"].unique())
    rows = sorted(table_data["Row"].unique())

    fig, ax = plt.subplots(figsize=(max(len(cols), 3), max(len(rows), 3)))
    ax.set_title(f"Maximum height check: table {table}")

    for _, row in table_data.iterrows():
        col_idx = cols.index(row["Column"])
        row_idx = rows.index(row["Row"])
        color = "red" if row["LargeHeight"] else "green"
        ax.add_patch(plt.Rectangle((col_idx, row_idx), 1, 1, color=color))
        ax.text(col_idx + 0.5, row_idx + 0.5, row["SampleName"], ha="center", va="center", color="white", fontsize=8)

    ax.set_xlim(0, len(cols))
    ax.set_ylim(0, len(rows))
    ax.set_xticks(np.arange(len(cols)) + 0.5)
    ax.set_xticklabels(cols)
    ax.set_yticks(np.arange(len(rows)) + 0.5)
    ax.set_yticklabels(rows)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.grid(True)

    red_patch = mpatches.Patch(color="red", label="Max height > 200")
    green_patch = mpatches.Patch(color="green", label="Max height <= 200")
    ax.legend(handles=[red_patch, green_patch], loc="upper right")

    plt.tight_layout()
    plt.show()
